# Phase 7 — Machine Learning Preprocessing

## Real Estate Investment Advisor

This notebook prepares the final modeling dataset for classification and regression.

### Objectives
- Load the final modeling dataset
- Define classification and regression targets
- Remove identifiers and target-generation helper columns
- Remove the raw multi-value `Amenities` field
- Create train/test splits
- Build numerical and categorical preprocessing pipelines
- Fit preprocessing on training data only
- Transform classification and regression data
- Save all preprocessing artifacts


In [21]:
import os
import joblib
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

print("Libraries imported successfully.")

Libraries imported successfully.


In [22]:
modeling_path = "../data/processed/real_estate_modeling_dataset.csv"

df = pd.read_csv(modeling_path)

print("Dataset loaded successfully.")
print("Shape:", df.shape)
print("Missing values:", df.isnull().sum().sum())
print("Duplicate rows:", df.duplicated().sum())

Dataset loaded successfully.
Shape: (250000, 34)
Missing values: 0
Duplicate rows: 0


In [23]:
print("Classification target:")
print(df["Good_Investment"].value_counts())

print("\nRegression target:")
print(df["Future_Price_5Y"].describe())

Classification target:
Good_Investment
1    130313
0    119687
Name: count, dtype: int64

Regression target:
count    250000.000000
mean        374.071613
std         207.689408
min          14.693281
25%         194.759437
50%         373.018319
75%         553.760366
max         734.664038
Name: Future_Price_5Y, dtype: float64


In [24]:
# Define targets and validate target values

classification_target = "Good_Investment"
regression_target = "Future_Price_5Y"

target_valid_mask = (
    df[classification_target].notna()
    & df[regression_target].notna()
)

removed_target_rows = int((~target_valid_mask).sum())

if removed_target_rows > 0:
    print("Removing rows with missing target values:", removed_target_rows)
    df = df.loc[target_valid_mask].copy()

y_classification = df[classification_target]
y_regression = df[regression_target]

print("Targets defined successfully.")
print("Dataset after target validation:", df.shape)
print("Missing Good_Investment:", int(y_classification.isna().sum()))
print("Missing Future_Price_5Y:", int(y_regression.isna().sum()))

Targets defined successfully.
Dataset after target validation: (250000, 34)
Missing Good_Investment: 0
Missing Future_Price_5Y: 0


In [25]:
# Remove identifiers, target columns and target-generation helper columns

columns_to_exclude = [
    "ID",
    "Investment_Price_Criterion",
    "Investment_PricePerSqFt_Criterion",
    "Investment_Amenity_Criterion",
    "Investment_Score",
    "Good_Investment",
    "Future_Price_5Y",
    "Amenities"
]

X = df.drop(columns=columns_to_exclude)

print("Feature matrix created.")
print("Shape:", X.shape)
print("\nFeatures:")
print(X.columns.tolist())

Feature matrix created.
Shape: (250000, 26)

Features:
['State', 'City', 'Locality', 'Property_Type', 'BHK', 'Size_in_SqFt', 'Price_in_Lakhs', 'Price_per_SqFt', 'Year_Built', 'Furnished_Status', 'Floor_No', 'Total_Floors', 'Age_of_Property', 'Nearby_Schools', 'Nearby_Hospitals', 'Public_Transport_Accessibility', 'Parking_Space', 'Security', 'Facing', 'Owner_Type', 'Availability_Status', 'Amenity_Density_Score', 'Price_per_BHK', 'Size_per_BHK', 'Property_Age_Category', 'Floor_Consistency']


In [26]:
# Identify numerical and categorical features

numerical_features = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object", "str"]
).columns.tolist()

print("Number of numerical features:", len(numerical_features))
print("Number of categorical features:", len(categorical_features))

print("\nNumerical features:")
print(numerical_features)

print("\nCategorical features:")
print(categorical_features)

Number of numerical features: 13
Number of categorical features: 13

Numerical features:
['BHK', 'Size_in_SqFt', 'Price_in_Lakhs', 'Price_per_SqFt', 'Year_Built', 'Floor_No', 'Total_Floors', 'Age_of_Property', 'Nearby_Schools', 'Nearby_Hospitals', 'Amenity_Density_Score', 'Price_per_BHK', 'Size_per_BHK']

Categorical features:
['State', 'City', 'Locality', 'Property_Type', 'Furnished_Status', 'Public_Transport_Accessibility', 'Parking_Space', 'Security', 'Facing', 'Owner_Type', 'Availability_Status', 'Property_Age_Category', 'Floor_Consistency']


In [27]:
# Classification train-test split

X_train_cls, X_test_cls, y_train_cls, y_test_cls = train_test_split(
    X,
    y_classification,
    test_size=0.20,
    random_state=42,
    stratify=y_classification
)

print("Classification split complete.")
print("Training features:", X_train_cls.shape)
print("Testing features :", X_test_cls.shape)
print("Training target  :", y_train_cls.shape)
print("Testing target   :", y_test_cls.shape)

Classification split complete.
Training features: (200000, 26)
Testing features : (50000, 26)
Training target  : (200000,)
Testing target   : (50000,)


In [28]:
# Regression train-test split

X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X,
    y_regression,
    test_size=0.20,
    random_state=42
)

print("Regression split complete.")
print("Training features:", X_train_reg.shape)
print("Testing features :", X_test_reg.shape)
print("Training target  :", y_train_reg.shape)
print("Testing target   :", y_test_reg.shape)

Regression split complete.
Training features: (200000, 26)
Testing features : (50000, 26)
Training target  : (200000,)
Testing target   : (50000,)


In [29]:
# Numerical preprocessing pipeline

numerical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

print("Numerical preprocessing pipeline created.")

Numerical preprocessing pipeline created.


In [30]:
# Categorical preprocessing pipeline

categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=True
            )
        )
    ]
)

print("Categorical preprocessing pipeline created.")

Categorical preprocessing pipeline created.


In [31]:
# Combine numerical and categorical pipelines

preprocessor = ColumnTransformer(
    transformers=[
        ("numerical", numerical_pipeline, numerical_features),
        ("categorical", categorical_pipeline, categorical_features)
    ]
)

print("Combined preprocessing pipeline created.")

Combined preprocessing pipeline created.


In [32]:
# Fit preprocessing using classification training data only

preprocessor.fit(X_train_cls)

print("Preprocessor fitted successfully.")

Preprocessor fitted successfully.


In [33]:
# Transform classification data

X_train_cls_processed = preprocessor.transform(X_train_cls)
X_test_cls_processed = preprocessor.transform(X_test_cls)

print("Classification preprocessing complete.")
print("Processed training shape:", X_train_cls_processed.shape)
print("Processed testing shape :", X_test_cls_processed.shape)

Classification preprocessing complete.
Processed training shape: (200000, 604)
Processed testing shape : (50000, 604)


In [34]:
# Transform regression data

X_train_reg_processed = preprocessor.transform(X_train_reg)
X_test_reg_processed = preprocessor.transform(X_test_reg)

print("Regression preprocessing complete.")
print("Processed training shape:", X_train_reg_processed.shape)
print("Processed testing shape :", X_test_reg_processed.shape)

Regression preprocessing complete.
Processed training shape: (200000, 604)
Processed testing shape : (50000, 604)


In [35]:
# Verify processed matrix types

print("Classification training matrix:", type(X_train_cls_processed))
print("Classification testing matrix :", type(X_test_cls_processed))
print("Regression training matrix    :", type(X_train_reg_processed))
print("Regression testing matrix     :", type(X_test_reg_processed))

Classification training matrix: <class 'scipy.sparse._csr.csr_matrix'>
Classification testing matrix : <class 'scipy.sparse._csr.csr_matrix'>
Regression training matrix    : <class 'scipy.sparse._csr.csr_matrix'>
Regression testing matrix     : <class 'scipy.sparse._csr.csr_matrix'>


In [36]:
# Create models directory

os.makedirs("../models", exist_ok=True)

print("Models directory ready.")

Models directory ready.


In [37]:
# Save fitted preprocessor

joblib.dump(
    preprocessor,
    "../models/preprocessor.pkl"
)

print("Preprocessor saved successfully.")
print("../models/preprocessor.pkl")

Preprocessor saved successfully.
../models/preprocessor.pkl


In [38]:
# Save classification processed data

joblib.dump(
    X_train_cls_processed,
    "../models/X_train_cls_processed.pkl"
)

joblib.dump(
    X_test_cls_processed,
    "../models/X_test_cls_processed.pkl"
)

joblib.dump(
    y_train_cls,
    "../models/y_train_cls.pkl"
)

joblib.dump(
    y_test_cls,
    "../models/y_test_cls.pkl"
)

print("Classification data saved successfully.")

Classification data saved successfully.


In [39]:
# Save regression processed data

joblib.dump(
    X_train_reg_processed,
    "../models/X_train_reg_processed.pkl"
)

joblib.dump(
    X_test_reg_processed,
    "../models/X_test_reg_processed.pkl"
)

joblib.dump(
    y_train_reg,
    "../models/y_train_reg.pkl"
)

joblib.dump(
    y_test_reg,
    "../models/y_test_reg.pkl"
)

print("Regression data saved successfully.")

Regression data saved successfully.


In [40]:
# Final verification

required_files = [
    "../models/preprocessor.pkl",
    "../models/X_train_cls_processed.pkl",
    "../models/X_test_cls_processed.pkl",
    "../models/y_train_cls.pkl",
    "../models/y_test_cls.pkl",
    "../models/X_train_reg_processed.pkl",
    "../models/X_test_reg_processed.pkl",
    "../models/y_train_reg.pkl",
    "../models/y_test_reg.pkl"
]

print("=" * 60)
print("PHASE 7 — ML PREPROCESSING COMPLETE")
print("=" * 60)

print("Dataset used:", df.shape)
print("Feature matrix:", X.shape)

print("\nClassification processed:")
print("Train:", X_train_cls_processed.shape)
print("Test :", X_test_cls_processed.shape)

print("\nRegression processed:")
print("Train:", X_train_reg_processed.shape)
print("Test :", X_test_reg_processed.shape)

print("\nTarget NaN checks:")
print("Classification train:", int(y_train_cls.isna().sum()))
print("Classification test :", int(y_test_cls.isna().sum()))
print("Regression train    :", int(y_train_reg.isna().sum()))
print("Regression test     :", int(y_test_reg.isna().sum()))

print("\nArtifact verification:")
for file_path in required_files:
    status = "OK" if os.path.exists(file_path) else "MISSING"
    print(f"{status:8} {file_path}")

PHASE 7 — ML PREPROCESSING COMPLETE
Dataset used: (250000, 34)
Feature matrix: (250000, 26)

Classification processed:
Train: (200000, 604)
Test : (50000, 604)

Regression processed:
Train: (200000, 604)
Test : (50000, 604)

Target NaN checks:
Classification train: 0
Classification test : 0
Regression train    : 0
Regression test     : 0

Artifact verification:
OK       ../models/preprocessor.pkl
OK       ../models/X_train_cls_processed.pkl
OK       ../models/X_test_cls_processed.pkl
OK       ../models/y_train_cls.pkl
OK       ../models/y_test_cls.pkl
OK       ../models/X_train_reg_processed.pkl
OK       ../models/X_test_reg_processed.pkl
OK       ../models/y_train_reg.pkl
OK       ../models/y_test_reg.pkl


# Phase 7 — Preprocessing Summary

The final modeling dataset was prepared for machine learning.

### Classification
Target: `Good_Investment`

An 80/20 stratified train-test split was created.

### Regression
Target: `Future_Price_5Y`

An 80/20 train-test split was created.

### Features Removed
- `ID`
- Investment target-generation criteria
- `Investment_Score`
- Target variables
- Raw multi-value `Amenities` field

### Numerical Preprocessing
- Median imputation
- StandardScaler

### Categorical Preprocessing
- Most-frequent imputation
- OneHotEncoder
- `handle_unknown="ignore"`

### Leakage Prevention
Target-generation helper columns were excluded from the predictor matrix.

The preprocessor was fitted using training data only.

### Saved Artifacts
- `models/preprocessor.pkl`
- Classification processed datasets and targets
- Regression processed datasets and targets
